# Case Study 6: Waste and Trash Detection with YOLOv8m

**Goal:** find the waste items in a photo, draw a box around each one, and name its category (Bottle, Can, Cardboard, Cup, Plastic, …).

**Steps:** 1. Setup → 2. Load the dataset → 3. Split it → 4. Convert to YOLO format → 5. Train YOLOv8m → 6. Test on unseen images → 7. Error analysis → 8. Try new images

**Before you start:** put this notebook, `yolov8m.pt` and the extracted dataset in one project folder:
```
waste-detection/
├── waste_detection_yolov8m.ipynb
├── yolov8m.pt
└── data/raw/          <- the extracted Kaggle dataset
```
Select the `.venv` kernel (top-right in VS Code), then run the cells from top to bottom with **Shift + Enter**.

## 1. Setup
- **ultralytics:** trains and evaluates YOLOv8.
- **torch:** runs the neural network, on the GPU if there is one.
- **PIL:** opens and saves images.
- **pandas:** builds the results tables.

In [ ]:
import random, shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import pandas as pd
import torch
from PIL import Image, ImageOps, ImageDraw
from IPython.display import Image as Show, display
from ultralytics import YOLO

print("GPU available:", torch.cuda.is_available())

## 2. Load the dataset
The Kaggle dataset has the same photos in several versions: **Original** (163 images), *Resized*, *Augmented* (4,923 images) and a ready-made *Split*.

We use the **163 original images** and their **Pascal VOC `.xml`** annotations, for two reasons:
- **No data leakage.** The augmented images are edited copies of the originals. If one copy of a photo lands in training and another copy in testing, the test score looks better than it really is. So we split the originals first, and YOLO augments the training images by itself during training.
- **VOC files store class names as text** (e.g. `<name>Bottle</name>`), so class IDs can't get mixed up.

The cell below finds the folder named "Original…" inside `DATASET_DIR`. If it prints `None`, set `ORIGINAL_DIR` by hand: right-click the folder in VS Code → *Copy Path*.

In [ ]:
DATASET_DIR  = Path("data/raw")            # where you extracted the Kaggle dataset
ORIGINAL_DIR = min((p for p in DATASET_DIR.rglob("*") if p.is_dir() and "original" in p.name.lower()),
                   key=lambda p: len(p.parts), default=None)
# ORIGINAL_DIR = Path(r"C:\path\to\Original Data")    # <- use this line instead if the one above gives None
print("Using:", ORIGINAL_DIR)

### Read the annotations
Each `.xml` file describes one image: its size, plus every object with a class name and a box in **pixels** (`xmin, ymin, xmax, ymax` = top-left and bottom-right corners).

In [ ]:
images = {p.stem: p for p in ORIGINAL_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

samples = []
for xml_file in ORIGINAL_DIR.rglob("*.xml"):
    if xml_file.stem not in images:
        continue
    root = ET.parse(xml_file).getroot()
    boxes = [(obj.find("name").text.strip().title(),       # "bottle cap " -> "Bottle Cap"
              *[float(obj.find(f"bndbox/{k}").text) for k in ("xmin", "ymin", "xmax", "ymax")])
             for obj in root.iter("object")]
    w, h = int(float(root.findtext("size/width", "0"))), int(float(root.findtext("size/height", "0")))
    if not (w and h):                                   # size missing in the xml -> read it from the image
        w, h = Image.open(images[xml_file.stem]).size
    samples.append({"img": images[xml_file.stem], "w": w, "h": h, "boxes": boxes})

CLASSES = sorted({b[0] for s in samples for b in s["boxes"]})
print(len(samples), "images,", sum(len(s["boxes"]) for s in samples), "objects")
print(len(CLASSES), "classes:", CLASSES)

### Look at the class balance
Some classes have far fewer examples than others, and rare classes usually score lower. This is why we report results **per class** as well as overall.

In [ ]:
counts = Counter(b[0] for s in samples for b in s["boxes"])
pd.Series(counts).sort_values().plot.barh(title="Objects per class", figsize=(7, 4));

## 3. Split into train / validation / test (70 / 15 / 15)
- **Train:** the images the model learns from.
- **Validation:** checked after every epoch to keep the best version of the model and to stop training early.
- **Test:** used **only once, at the end**. These are the "unseen images" the brief asks for.

We split by **image** (all boxes of a photo stay together). We sort the images first and use a fixed random seed, so the split comes out the same every time and on every teammate's laptop.

In [ ]:
samples.sort(key=lambda s: s["img"].name)
random.seed(42)
random.shuffle(samples)

n_train, n_val = int(0.70 * len(samples)), int(0.15 * len(samples))
splits = {"train": samples[:n_train],
          "val":   samples[n_train:n_train + n_val],
          "test":  samples[n_train + n_val:]}
print({name: len(items) for name, items in splits.items()})

## 4. Convert to YOLO format
YOLO needs one `.txt` label file per image, with one line per object:
```
class_id  x_center  y_center  width  height        e.g.  2 0.51 0.43 0.22 0.35
```
All four numbers are **divided by the image width or height**, so they are between 0 and 1. We also clip boxes to the image edges; YOLO skips any label that goes past them.

**Phone-photo rotation:** phones often save photos sideways and add a "rotate me" tag (EXIF). If the XML's width and height are swapped compared with the saved pixels, we rotate the image to match what the annotator saw, then save it without the tag.

In [ ]:
YOLO_DIR = Path("data/trash_yolo")
shutil.rmtree(YOLO_DIR, ignore_errors=True)            # start fresh every time

for split, items in splits.items():
    (YOLO_DIR / "images" / split).mkdir(parents=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True)
    for s in items:
        img = Image.open(s["img"])
        if (s["w"], s["h"]) == (img.height, img.width) and s["w"] != s["h"]:
            img = ImageOps.exif_transpose(img)          # fix sideways phone photo
        img.convert("RGB").save(YOLO_DIR / "images" / split / f"{s['img'].stem}.jpg", quality=95)

        lines = []
        for name, x1, y1, x2, y2 in s["boxes"]:
            x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, s["w"]), min(y2, s["h"])   # keep inside image
            lines.append(f"{CLASSES.index(name)} {(x1 + x2) / 2 / s['w']:.6f} {(y1 + y2) / 2 / s['h']:.6f} "
                         f"{(x2 - x1) / s['w']:.6f} {(y2 - y1) / s['h']:.6f}")
        (YOLO_DIR / "labels" / split / f"{s['img'].stem}.txt").write_text("\n".join(lines))

# data.yaml tells YOLO where the images are and what the classes are called
DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text(f"path: {YOLO_DIR.resolve().as_posix()}\n"
                     "train: images/train\nval: images/val\ntest: images/test\n"
                     f"names: {CLASSES}\n")
print(DATA_YAML.read_text())

### Check the labels before training (good practice)
We draw the converted boxes back onto 3 images. If the boxes sit neatly on the objects, the conversion is correct.

In [ ]:
for img_path in sorted((YOLO_DIR / "images" / "train").glob("*.jpg"))[:3]:
    img = Image.open(img_path)
    draw, (W, H) = ImageDraw.Draw(img), img.size
    for line in (YOLO_DIR / "labels" / "train" / f"{img_path.stem}.txt").read_text().splitlines():
        c, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = (xc - bw / 2) * W, (yc - bh / 2) * H
        draw.rectangle([x1, y1, x1 + bw * W, y1 + bh * H], outline="red", width=max(2, W // 200))
        draw.text((x1, y1), CLASSES[int(c)], fill="red", font_size=max(12, W // 30))
    img.thumbnail((500, 500))
    display(img)

## 5. Train YOLOv8m
**How YOLOv8m works:** YOLO ("You Only Look Once") is a **one-stage detector**. One pass through the network predicts all boxes and classes at once, which makes it fast.
- **Backbone:** extracts features (edges → textures → object parts).
- **Neck:** combines features at 3 scales, so small, medium and large objects can all be found.
- **Head:** predicts the class and box for each object. It is *anchor-free*: it predicts box edges directly instead of adjusting preset box shapes.
- **NMS (Non-Maximum Suppression):** removes duplicate boxes on the same object and keeps the one with the highest confidence.
- "m" = medium size (about 26 M parameters).

**Training concepts:**
- **Transfer learning:** `yolov8m.pt` was already trained on **COCO** (118k images, 80 everyday objects, including *bottle* and *cup*). We fine-tune it on our 8 classes. That's why ~115 training images are enough.
- **Epoch:** one pass over all training images. **Batch:** images per weight update. **imgsz:** size images are resized to.
- **Augmentation (automatic):** mosaic (4 images stitched into one), flips, colour changes and scaling. The model sees new versions of the training images in every epoch, which helps with the "different shapes, sizes and backgrounds" challenge.
- **Early stopping (`patience`):** stops training when validation mAP stops improving. The best epoch is saved as `best.pt`.
- **Loss:** box loss (how well the boxes overlap) + class loss (right category?) + DFL (sharper box edges). Training adjusts the weights to make the loss smaller.

The CPU is much slower than a GPU, so without a GPU we train for fewer epochs at a smaller image size.

In [ ]:
GPU    = torch.cuda.is_available()
EPOCHS = 50 if GPU else 20
IMGSZ  = 640 if GPU else 480

model = YOLO("yolov8m.pt")                 # COCO-pretrained weights
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=8,                                # lower to 4 if you get "CUDA out of memory"
    patience=15,                            # early stopping
    seed=42,                                # reproducible results
    name="yolov8m", exist_ok=True,          # results go to runs/detect/yolov8m
)
RUN_DIR = Path(model.trainer.save_dir)      # folder where YOLO saved weights, plots and results.csv
print("Saved to:", RUN_DIR)

### Training curves
`results.png` (saved by YOLO) shows every epoch. The **losses should go down** and **precision, recall and mAP should go up**, then level off.
If validation loss starts rising while training loss keeps falling, the model is **overfitting**, i.e. memorising the training images.

In [ ]:
display(Show(filename=str(RUN_DIR / "results.png"), width=900))
display(Show(filename=str(RUN_DIR / "train_batch0.jpg"), width=600))   # a training batch after mosaic augmentation

## 6. Evaluate on the unseen TEST images
**Key terms:**
- **IoU (Intersection over Union):** overlap area ÷ combined area of the predicted and true box. A detection is counted correct if IoU ≥ 0.5 and the class is right.
- **Precision:** of all detections, how many were correct? (few false alarms)
- **Recall:** of all real objects, how many were found? (few misses)
- **AP:** area under the precision-recall curve for one class, which combines precision and recall at every confidence level.
- **mAP@0.5:** AP averaged over all classes, with IoU ≥ 0.5 counted as correct.
- **mAP@0.5:0.95:** mAP averaged over stricter IoU thresholds (0.5 to 0.95), so it rewards **tight, accurate boxes**. This measures bounding-box quality.

In [ ]:
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))       # the best epoch from training
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ,
                   plots=True,                           # also builds the confusion matrix used in step 7
                   name="yolov8m_test", exist_ok=True)
TEST_DIR = Path(metrics.save_dir)

print(f"\nPrecision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")
print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")

### Results per class
Which classes are easy and which are hard? Small or rare items (bottle caps, cigarette butts) usually score lowest.

In [ ]:
per_class = pd.DataFrame({
    "class":     [CLASSES[i] for i in metrics.ap_class_index],
    "precision": metrics.box.p,
    "recall":    metrics.box.r,
    "AP@0.5":    metrics.box.ap50,
    "AP@0.5:0.95": metrics.box.ap,
}).round(3)
per_class.to_csv("results_per_class.csv", index=False)
per_class

## 7. Error analysis: correct, missed and false detections
YOLO's **confusion matrix** counts every detection with confidence ≥ 0.25. Rows are the **predicted** class, columns the **true** class, plus an extra **background** row and column:
- **Diagonal:** correct detections.
- **Background row:** a real object that wasn't detected (**missed**, false negative).
- **Background column:** a detection where there is no object (**false alarm**, false positive).
- **Other off-diagonal cells:** the object was found but given the **wrong class** (e.g. Plastic called Bottle).

Note: "false alarm" is counted under the class the model *predicted*. The other columns are counted under the *true* class.

In [ ]:
display(Show(filename=str(TEST_DIR / "confusion_matrix.png"), width=700))

cm = metrics.confusion_matrix.matrix       # [predicted, true]; last row/column = background
n = len(CLASSES)
errors = pd.DataFrame({
    "correct":          cm.diagonal()[:n],
    "missed":           cm[n, :n],
    "wrong class":      cm[:n, :n].sum(axis=0) - cm.diagonal()[:n],
    "false alarm":      cm[:n, n],
}, index=CLASSES).astype(int)
errors.loc["TOTAL"] = errors.sum()
errors.to_csv("error_analysis.csv")
errors

### See the errors on the test images
YOLO saved the **ground truth** (`labels`) and the **predictions** (`pred`) for the test images. Compare each pair: a box missing from `pred` is a missed object, and an extra box is a false alarm.

In [ ]:
for labels_img in sorted(TEST_DIR.glob("val_batch*_labels.jpg")):
    print(labels_img.name, " vs ", labels_img.name.replace("labels", "pred"))
    display(Show(filename=str(labels_img), width=520), Show(filename=str(labels_img).replace("labels", "pred"), width=520))

## 8. Try the model on new images
Put your own photos of rubbish in the `my_test_images` folder, then run this cell. It makes a good live demo for the evaluation. If the folder is empty, it runs on 3 test images instead.

In [ ]:
MY_IMAGES = Path("my_test_images")
MY_IMAGES.mkdir(exist_ok=True)
files = sorted(p for p in MY_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) \
        or sorted((YOLO_DIR / "images" / "test").glob("*.jpg"))[:3]

for r in best.predict(source=[str(f) for f in files], conf=0.25, imgsz=IMGSZ, verbose=False):
    out = Image.fromarray(r.plot()[:, :, ::-1])         # draw boxes; BGR -> RGB
    out.thumbnail((700, 700))
    print(Path(r.path).name, "->", len(r.boxes), "objects")
    display(out)

## 9. Conclusion (write this in your report)
Use the numbers above (also saved as `results_per_class.csv` and `error_analysis.csv`) together with the plots in `runs/detect/yolov8m/` (training) and `runs/detect/yolov8m_test/` (test):
1. **Overall result:** precision, recall, mAP@0.5 and mAP@0.5:0.95 on the test set.
2. **Best and worst classes:** why? (size, number of examples, look-alike classes)
3. **Error types:** mostly missed objects, false alarms or wrong classes? Relate this to the challenges in the brief (overlapping objects, different shapes, sizes and backgrounds).
4. **Limitations:** only 163 images and a small test set of ~25 images.
5. **Improvements:** more images, more epochs, higher image size for small objects, tuning the confidence threshold.

### Quick viva answers
| Question | Answer |
|---|---|
| Why not use the augmented images? | They are copies of the originals, so the same photo could appear in both train and test (data leakage) and inflate the scores. |
| Why separate validation and test sets? | Validation picks the best epoch, so we tune on it. Test is used once, which gives an honest score on unseen data. |
| What is transfer learning? | Starting from weights already trained on COCO and fine-tuning them on our images. |
| What is IoU? | Overlap ÷ union of two boxes. We count a detection as correct at IoU ≥ 0.5. |
| mAP@0.5 vs mAP@0.5:0.95? | The first checks "found it". The second also rewards tight boxes. |
| What is NMS? | It removes duplicate overlapping boxes for the same object and keeps the most confident one. |
| What does the confidence threshold do? | Higher threshold means fewer false alarms but more misses (precision ↑, recall ↓). |
| How would you deploy this? | Export `best.pt` (e.g. `best.export(format="onnx")`) and run it on a camera above the conveyor belt to trigger the right sorting bin. |